# Create semantic queries of varying topic, length, complexity, and filters for semantic search benchmarking

# Import Libraries

In [1]:
import numpy as np
import pandas as pd

import ast
import random
random.seed(1234)

from datetime import datetime, timedelta
from urllib.parse import parse_qs, unquote_plus, urlencode, quote_plus, quote

# Load the CL query samples

Selected from CLReplica

```
SELECT id, source, engine, get_params
FROM public.search_searchquery
ORDER BY random()
LIMIT 10000;
```

In [30]:
df = pd.read_csv("queries.csv")

## Filter for engine 1 from elasticsearch

In [3]:
df = df[df["engine"] == 1]
len(df)

9628

## Parse params

In [31]:
# function to parse a single params string into a dict
def parse_params(param_str):
    decoded = unquote_plus(param_str)
    return {k: v[0] if len(v) == 1 else v for k, v in parse_qs(decoded).items()}

# parse get_params
parsed = df["get_params"].apply(parse_params)
parsed_df = pd.json_normalize(parsed)
df_parsed = pd.concat([df, parsed_df], axis=1)

## Filter for only type==o or type is missing for case law searches

In [6]:
df_parsed = df_parsed[(df_parsed["type"] == "o") | (df_parsed["type"].isnull())]
len(df_parsed)

6389

## Remove records that do not have q param (for extracting keywords)

In [7]:
df_filtered = df_parsed[~df_parsed["q"].isnull()]
len(df_filtered)

5523

## Get keywords
1.	cites:(123...), cites:(123 OR 456 ...)
2.	related:123456
3.	Dashed numbers like 17-14-9, 2021-CR-1234
4.	Decimal numbers like 628.071
5.	Case names in the form "xxx v. yyy" or "Smith v. Jones"
6.	Key-value pairs like id:68954, court:ny
7.  Citation-looking patterns like: "585 U.S. 961"

In [8]:
pattern = (
    r'cites:\(\d+(?:\s+OR\s+\d+)*\)'            # cites:(123) or cites:(123 OR 456)
    r'|related:\d+'                             # related:123
    r'|\b\d+-\d+(?:-\d+)*\b'                    # 17-14-9 or 2021-CR-1234
    r'|\b\d+\.\d+\b'                            # decimal like 628.071
    r'|\b\w[\w\.\-]*\s+v\.\s+\w[\w\.\-]*\b'     # case name like Smith v. Jones
    r'|\bid:\d+\b'                              # id:12345
    r'|\bcourt:\w+\b'                           # court:ny
    r'|\bcourt_id:\w+\b'                         # court:ny
)

df_filtered["q"] = df_filtered["q"].astype(str)
mask = df_filtered["q"].str.contains(pattern, regex=True, na=False)
df_filtered = df_filtered[~mask]
len(df_filtered)

/var/folders/d9/3h7m7wc52kv6fgxmbyd8s0940000gn/T/ipykernel_45144/3623288734.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered["q"] = df_filtered["q"].astype(str)


514

In [9]:
df_filtered["q"].to_csv("keywords.csv", index=False)

## Load cl courts data to get court ids for filtering

In [10]:
courts_df = pd.read_json("courts.json")
courts_df = courts_df[courts_df["in_use"] == True]
courts_options = list(courts_df["id"].unique())
len(courts_options)

470

# Load cl judges data to get judges for filtering

The cl judges data is queried from CLReplica using below syntax

```
def sample_judges():
    judges = []
    person_ids = list(Person.objects.all())
    sampled_persons = random.sample(person_ids, 20)
    for person in sampled_persons:
        if person.is_judge:
            judges.append(person.name_last)

    sampled_persons = random.sample(person_ids, 20)
    for person in sampled_persons:
        if person.is_judge:
            judges.append(person.name_full)

    print(len(judges))
    print(judges)
```

In [11]:
judges = list(set(['Murphy', 'Foley', 'Haynes', 'Moss', 'Siwek', 'Lynch', 'Kevins', 'Bye', 'Hummel', 'Halyburton', 'Naughton', 'Henry', 'Breading', 'Sheldon', 'Brown', 'Davis', 'Collins', 'Youngblood', 'Hillman', 'Nash', 'Deborah Agosti', 'Peter Grometer', 'Claire Eagan', 'John Wallace Winborne', 'Isaac N. Sullivan', 'Arthur  W. Jones', 'Leonard Livote', 'James  L. McKechnie', 'William Mills Maltbie', 'John Weld Peck', 'Eli Shelby Hammond', 'Yvonne Theresa Sanchez', 'Robert B. Collings', 'Bahatti E. Pitt', 'John  J. Lynch', 'Gale  E. Kaneshiro', 'Lawrence Edward Ornell', 'Thomas J. Tucker', 'Kirtan K. Khalsa']))
len(judges)

39

# Load the semantic queries generated using GPT 4.1 mini

In [12]:
queries = pd.read_csv("gpt_queries.csv")
queries.head()

,keywords,included_words,query
0,"['HIPPA', 'insurance']",['HIPPA'],"How does ""HIPPA"" protect patient information w..."
1,['tax preparation services'],[],What are the legal requirements and consumer p...
2,['personal tax return'],[],What are the legal requirements and deadlines ...
3,['warranty of habitability'],[],What does the warranty of habitability require...
4,['illegal contract'],[],What constitutes an illegal contract under U.S...


# For each semantic query, set-up 5 filter scenarios (1 without any filters and 4 with randomly selected filters)

In [13]:
filters = {"courts": ["court"], # court=ca4+ca5
           "judges": ["judge"], # judge=brown
           "stats": ["stat_Published", "stat_Unpublished", "stat_Errata", "stat_Separate", "stat_In-chambers", 'stat_Relating-to', 'stat_Unknown'], # "stat_Published=on"
           "filed_before": ["filed_before"], # filed_before=01/01/2025
           "filed_after": ["filed_after"], # filed_after=01/01/2025
           "min_cites": ["cited_gt"], # cited_gt=20
           "max_cites": ["cited_lt"] # cited_lt=100
}

In [14]:
options = [item for sublist in filters.values() for item in sublist]
options

['court',
 'judge',
 'stat_Published',
 'stat_Unpublished',
 'stat_Errata',
 'stat_Separate',
 'stat_In-chambers',
 'stat_Relating-to',
 'stat_Unknown',
 'filed_before',
 'filed_after',
 'cited_gt',
 'cited_lt']

In [15]:
def generate_filters():
    result = []
    for _ in range(4):
        k = random.randint(1, int(len(options)/2))
        result.append(random.sample(options, k))
    return result

queries["query_filters"] = [generate_filters() for _ in range(len(queries))]

In [16]:
queries.head()

,keywords,included_words,query,query_filters
0,"['HIPPA', 'insurance']",['HIPPA'],"How does ""HIPPA"" protect patient information w...","[[judge, court, cited_lt, filed_before], [file..."
1,['tax preparation services'],[],What are the legal requirements and consumer p...,"[[filed_before, stat_Relating-to, cited_lt, ci..."
2,['personal tax return'],[],What are the legal requirements and deadlines ...,"[[judge, filed_before, filed_after, cited_lt],..."
3,['warranty of habitability'],[],What does the warranty of habitability require...,"[[court], [stat_Published, cited_gt, stat_Rela..."
4,['illegal contract'],[],What constitutes an illegal contract under U.S...,"[[stat_Published, court], [cited_lt, judge, st..."


# Construct the api queries using the semantic queries and the query filters

In [17]:
def get_rand_date(start_date, end_date):    
    delta = (end_date - start_date).days
    random_days = random.randint(0, delta)
    random_date = start_date + timedelta(days=random_days)
    return random_date.date()


def encode_params_selectively(params):
    encoded_parts = []
    
    for key, value in params.items():
        if key == 'q':
            # Use quote for 'q' parameter (spaces become %20)
            encoded_value = quote(str(value))
        elif key == 'court':
            # For court parameter, encode everything except '+' signs
            # quote(safe='+') preserves '+' characters
            encoded_value = quote(str(value), safe='+')
        else:
            # For other parameters, use standard quote
            encoded_value = quote(str(value))
        
        encoded_parts.append(f"{key}={encoded_value}")
    
    return '&'.join(encoded_parts)


def construct_qpi_query(query, filters, courts=courts_options, judges=judges):
    root = "https://www.courtlistener.com/api/rest/v4/search/?type=o&semantic=true"
    params = {"q": query}

    # Random courts
    if "court" in filters:
        k = random.randint(1, int(len(courts)/4))
        court_filter = random.sample(courts, k)
        params["court"] = "+".join(court_filter)

    # Random judge
    if "judge" in filters:
        judge_filter = random.sample(judges, 1)
        params["judge"] = judge_filter[0]

    # Stat filters
    for s in filters:
        if s.startswith("stat_"):
            params[s] = "on"

    # Filing dates
    if "filed_before" in filters and "filed_after" in filters:
        before = get_rand_date(datetime(1900, 1, 1), datetime(2023, 8, 1))
        after = get_rand_date(datetime.strptime(before, "%Y-%m-%d"), datetime(2025, 8, 1))
        params["filed_before"] = before.strftime("%Y-%m-%d")
        params["filed_after"] = after.strftime("%Y-%m-%d")
    elif "filed_before" in filters:
        params["filed_before"] = get_rand_date(datetime(2000, 1, 1), datetime(2025, 8, 1)).strftime("%Y-%m-%d")
    elif "filed_after" in filters:
        params["filed_after"] = get_rand_date(datetime(1900, 1, 1), datetime(2023, 8, 1)).strftime("%Y-%m-%d")

    # Citation counts
    if "cited_gt" in filters and "cited_lt" in filters:
        cited_gt = random.randint(1, 30)
        cited_lt = random.randint(cited_gt, 200)
        params["cited_gt"] = cited_gt
        params["cited_lt"] = cited_lt
    elif "cited_gt" in filters:
        params["cited_gt"] = random.randint(1, 30)
    elif "cited_lt" in filters:
        params["cited_lt"] = random.randint(100, 200)

    query_string = encode_params_selectively(params)
    full_url = f"{root}&{query_string}"

    return full_url

In [18]:
for index, row in queries.iterrows():
    query = row["query"]
    queries.loc[index, "query_filter_none"] = construct_qpi_query(query, [])
    
    row_filters = queries["query_filters"].iloc[0]
    for i, filters in enumerate(row_filters):
        queries.loc[index, f"query_filter_{i}"] = construct_qpi_query(query, filters)

In [19]:
queries.head()

,keywords,included_words,query,query_filters,query_filter_none,query_filter_0,query_filter_1,query_filter_2,query_filter_3
0,"['HIPPA', 'insurance']",['HIPPA'],"How does ""HIPPA"" protect patient information w...","[[judge, court, cited_lt, filed_before], [file...",https://www.courtlistener.com/api/rest/v4/sear...,https://www.courtlistener.com/api/rest/v4/sear...,https://www.courtlistener.com/api/rest/v4/sear...,https://www.courtlistener.com/api/rest/v4/sear...,https://www.courtlistener.com/api/rest/v4/sear...
1,['tax preparation services'],[],What are the legal requirements and consumer p...,"[[filed_before, stat_Relating-to, cited_lt, ci...",https://www.courtlistener.com/api/rest/v4/sear...,https://www.courtlistener.com/api/rest/v4/sear...,https://www.courtlistener.com/api/rest/v4/sear...,https://www.courtlistener.com/api/rest/v4/sear...,https://www.courtlistener.com/api/rest/v4/sear...
2,['personal tax return'],[],What are the legal requirements and deadlines ...,"[[judge, filed_before, filed_after, cited_lt],...",https://www.courtlistener.com/api/rest/v4/sear...,https://www.courtlistener.com/api/rest/v4/sear...,https://www.courtlistener.com/api/rest/v4/sear...,https://www.courtlistener.com/api/rest/v4/sear...,https://www.courtlistener.com/api/rest/v4/sear...
3,['warranty of habitability'],[],What does the warranty of habitability require...,"[[court], [stat_Published, cited_gt, stat_Rela...",https://www.courtlistener.com/api/rest/v4/sear...,https://www.courtlistener.com/api/rest/v4/sear...,https://www.courtlistener.com/api/rest/v4/sear...,https://www.courtlistener.com/api/rest/v4/sear...,https://www.courtlistener.com/api/rest/v4/sear...
4,['illegal contract'],[],What constitutes an illegal contract under U.S...,"[[stat_Published, court], [cited_lt, judge, st...",https://www.courtlistener.com/api/rest/v4/sear...,https://www.courtlistener.com/api/rest/v4/sear...,https://www.courtlistener.com/api/rest/v4/sear...,https://www.courtlistener.com/api/rest/v4/sear...,https://www.courtlistener.com/api/rest/v4/sear...


## Inspect a few of the queries to ensure they work

In [20]:
queries["query_filter_none"].iloc[0]

'https://www.courtlistener.com/api/rest/v4/search/?type=o&semantic=true&q=How%20does%20%22HIPPA%22%20protect%20patient%20information%20when%20dealing%20with%20insurance%20companies%2C%20and%20what%20legal%20obligations%20do%20insurers%20have%20to%20comply%20with%20these%20privacy%20rules%3F'

In [21]:
queries["query_filter_0"].iloc[0]

'https://www.courtlistener.com/api/rest/v4/search/?type=o&semantic=true&q=How%20does%20%22HIPPA%22%20protect%20patient%20information%20when%20dealing%20with%20insurance%20companies%2C%20and%20what%20legal%20obligations%20do%20insurers%20have%20to%20comply%20with%20these%20privacy%20rules%3F&court=txeb+txwd+grrondect+flsd+nhd+californiad+oknd+texag+ccpa+vt+ilnd+almd+or+vtb+idb+mo+mieb+tennworkcompapp+acca+ncctapp+minnag+com+prsupreme+iowactapp+armfor+ca9+vawb+illinoisd+gud+msnb+txsb+iand+med+wisag+webchippewatr+bapme+alnb+coquct+asbca+ga+cnmisuperct+navajofamct+risuperct+tulalipctapp+grtravbandct+nevapp+tned+flmd+alsd+pamd+illappct+ohio+laag+indtc+grrondectapp+nceb+texjpml+mesuperct+njtaxct+ca8+flmb+flsb+alacivapp+wieb+scotus+alaskactapp+kanctapp+arizctapp+nynb+alsb+monttc+washag+idd+pawd&judge=Nash&filed_before=2023-12-30&cited_lt=191'

In [22]:
queries["query_filter_1"].iloc[0]

'https://www.courtlistener.com/api/rest/v4/search/?type=o&semantic=true&q=How%20does%20%22HIPPA%22%20protect%20patient%20information%20when%20dealing%20with%20insurance%20companies%2C%20and%20what%20legal%20obligations%20do%20insurers%20have%20to%20comply%20with%20these%20privacy%20rules%3F&filed_after=2014-07-06'

In [23]:
queries["query_filter_2"].iloc[0]

'https://www.courtlistener.com/api/rest/v4/search/?type=o&semantic=true&q=How%20does%20%22HIPPA%22%20protect%20patient%20information%20when%20dealing%20with%20insurance%20companies%2C%20and%20what%20legal%20obligations%20do%20insurers%20have%20to%20comply%20with%20these%20privacy%20rules%3F&court=ncwd+txsd+vawd+tennsuperct+cc+olc+nysurct+miss+mich+alsb+arb+idd+ncbizct+fla+vtb+mdctspecapp+sacfoxsupct+wvsb+kingsbench+oklaag+ohioctapp+bva+conn+azd+monttc+mohegangct+moag+okla+nyappdiv+mspb+hochunk+ilcb+ksd+alaska+nd+cafc+cnmisuperct+mdd+mc+uscgcoca+ca10+utd+calappdeptsuper+ariz+ared+ca6+wisctapp+casb+tenn+gud+nc+nysb+mnb+casd+neb+usarmymilrev+bta+wied+bap8+connsuperct+oknb+wiwb+dcd+washterr+laeb+nmib+ca9+txed+oneidactapp+nycrimct+cheyrsiouxctapp+usnmcmilrev+ctb+nceb+nydistct+tennworkcompcl+ftmcdowctapp+cadc+wva+nmid+montag+nywd+hid+nhb&judge=Claire%20Eagan&stat_Separate=on&stat_Unpublished=on&stat_Unknown=on&cited_lt=193'

In [24]:
queries["query_filter_3"].iloc[0]

'https://www.courtlistener.com/api/rest/v4/search/?type=o&semantic=true&q=How%20does%20%22HIPPA%22%20protect%20patient%20information%20when%20dealing%20with%20insurance%20companies%2C%20and%20what%20legal%20obligations%20do%20insurers%20have%20to%20comply%20with%20these%20privacy%20rules%3F&stat_Separate=on'

# Save the queries 

In [25]:
queries.to_csv("benchmark_queries.csv", index=False)

In [26]:
all_queries = []
for col in ['query_filter_none', 'query_filter_0', 'query_filter_1', 'query_filter_2', 'query_filter_3']:
    all_queries.extend(queries[col].to_list())
len(all_queries)

560

In [27]:
with open("benchmark_queries_all.txt", "w", encoding="utf-8") as f:
    for item in all_queries:
        f.write(f"{item}\n")

In [28]:
sampled_queries = random.sample(all_queries, 100)
len(sampled_queries)

100

In [29]:
with open("benchmark_queries_sample.txt", "w", encoding="utf-8") as f:
    for item in all_queries:
        f.write(f"{item}\n")